In [51]:
!git clone https://github.com/cpoiato/HYDROSIGHT.git /content/HYDROSIGHT

fatal: destination path '/content/HYDROSIGHT' already exists and is not an empty directory.


In [52]:
%cd /content/HYDROSIGHT
!git pull --ff-only

/content/HYDROSIGHT
Already up to date.


In [53]:
%cd /content/HYDROSIGHT/src/hydrosight/eda/notebooks

/content/HYDROSIGHT/src/hydrosight/eda/notebooks


In [54]:
from pathlib import Path

repo_path = Path("/content/HYDROSIGHT")

print("Repository folder exists:", repo_path.exists())
print("Git metadata exists:", (repo_path / ".git").exists())
print("Current directory:", Path.cwd())

Repository folder exists: True
Git metadata exists: True
Current directory: /content/HYDROSIGHT/src/hydrosight/eda/notebooks


In [55]:
from google.colab import drive
import json

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [56]:
from pathlib import Path
import sys

ROOT_DIR = Path("/content/HYDROSIGHT")
sys.path.insert(0, str(ROOT_DIR / "src"))
RAW_DIR = ROOT_DIR / "data" / "raw" / "hospitals"
REPORT_DIR = ROOT_DIR / "reports" / "eda" / "results"
FIGURES_DIR = ROOT_DIR / "reports" / "eda" / "figures"


assert RAW_DIR.exists(), f"Folder not found: {RAW_DIR}"
assert RAW_DIR.exists(), f"Folder not found: {RAW_DIR}"

print("Project:", ROOT_DIR)
print("Raw data:", RAW_DIR)

print("\nAvailable pipe files:")
for file in sorted(RAW_DIR.glob("*.geojson")):
    print(file.name)

Project: /content/HYDROSIGHT
Raw data: /content/HYDROSIGHT/data/raw/hospitals

Available pipe files:
hospitals_20261024T153107Z.geojson


In [57]:
import pandas as pd
from IPython.display import display


files = sorted(RAW_DIR.glob("hospitals_*.geojson"))

assert files, "No hospitals snapshots found."

latest_file = files[-1]

df = pd.read_json(latest_file)
df.shape

(5, 2)

In [58]:
display("Dataset Information")
df.info()

'Dataset Information'

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   type      5 non-null      object
 1   features  5 non-null      object
dtypes: object(2)
memory usage: 212.0+ bytes


In [59]:
display("Sample Data")
display(df)

'Sample Data'

,type,features
0,FeatureCollection,"{'type': 'Feature', 'id': 167697, 'geometry': ..."
1,FeatureCollection,"{'type': 'Feature', 'id': 167711, 'geometry': ..."
2,FeatureCollection,"{'type': 'Feature', 'id': 167726, 'geometry': ..."
3,FeatureCollection,"{'type': 'Feature', 'id': 167750, 'geometry': ..."
4,FeatureCollection,"{'type': 'Feature', 'id': 167800, 'geometry': ..."


In [60]:
geometry = df["features"].iloc[0]

if isinstance(geometry, dict):
        print("Key:", geometry.keys())
        print("Type:", geometry.get("type"))
        print("Coordinates:", geometry.get("geometry"))
else:
        print("Type:", geometry.get("type"))
        print("Coordinates:", geometry.get("geometry"))

Key: dict_keys(['type', 'id', 'geometry', 'properties'])
Type: Feature
Coordinates: {'type': 'Point', 'coordinates': [-114.132190876209, 51.0641896775293]}


In [61]:
import geopandas as gpd

# Load Calgary city boundary from the official ArcGIS service
boundary_url = (
    "https://services1.arcgis.com/AVP60cs0Q9PEA8rH/"
    "arcgis/rest/services/City_Boundary/FeatureServer/0/query"
    "?where=1%3D1"
    "&outFields=*"
    "&outSR=4326"
    "&returnGeometry=true"
    "&f=geojson"
)

calgary_boundary = gpd.read_file(boundary_url)

# Validate the boundary geometry
print("Total records:", len(calgary_boundary))
print("CRS:", calgary_boundary.crs)
print("Geometry types:")
print(calgary_boundary.geometry.geom_type.value_counts())
print("Bounds:", calgary_boundary.total_bounds)
print("Valid geometries:", calgary_boundary.geometry.is_valid.sum())

assert (
    not calgary_boundary.empty
    and calgary_boundary.geometry.notna().all()
    and calgary_boundary.geometry.is_valid.all()
    and calgary_boundary.geometry.geom_type.isin(
        ["Polygon", "MultiPolygon"]
    ).all()
), "Invalid Calgary boundary geometry"

Total records: 1
CRS: EPSG:4326
Geometry types:
Polygon    1
Name: count, dtype: int64
Bounds: [-114.31580385   50.84282878 -113.85991285   51.21243208]
Valid geometries: 1


In [62]:
from shapely.geometry import shape

# Convert GeoJSON MultiLineString to Shapely geometry
hospitals_gdf = gpd.GeoDataFrame(
    df.copy(),
    geometry=df["features"].apply(
        lambda value: shape(json.loads(value))
        if isinstance(value, str) else shape(value)
    ),
    crs="EPSG:4326"
)

In [63]:
# Align coordinate reference systems
hospitals = hospitals_gdf.to_crs(calgary_boundary.crs)

# Merge municipal boundary geometries
city_boundary = calgary_boundary.geometry.union_all()

# Classify pipe locations
hospitals["boundary_status"] = "outside"

hospitals.loc[
    hospitals.geometry.intersects(city_boundary),
    "boundary_status"
] = "partially_inside"

hospitals.loc[
    hospitals.geometry.covered_by(city_boundary),
    "boundary_status"
] = "inside"

# Display validation results
print("Hospitals boundary validation results:")
print(hospitals["boundary_status"].value_counts())

Hospitals boundary validation results:
boundary_status
inside    5
Name: count, dtype: int64


In [64]:
print("Total records:", len(hospitals))
print("CRS:", hospitals.crs)
print("Geometry types:")
print(hospitals.geometry.geom_type.value_counts())
print("Bounds:", hospitals.total_bounds)
print("Valid geometries:", hospitals.geometry.is_valid.sum())
print("Null values:")
print(hospitals.isnull().sum())
print("Duplicate values (excluding 'features' column):")
# We exclude the 'features' column because dictionaries inside are unhashable
hashable_cols = [col for col in hospitals.columns if col != 'features']
print(hospitals.duplicated(subset=hashable_cols).sum())

Total records: 5
CRS: EPSG:4326
Geometry types:
Point    5
Name: count, dtype: int64
Bounds: [-114.14797189   50.8821086  -113.95171799   51.07902311]
Valid geometries: 5
Null values:
type               0
features           0
geometry           0
boundary_status    0
dtype: int64
Duplicate values (excluding 'features' column):
0


In [65]:
df_clean = df.copy()

In [66]:
from datetime import datetime, timezone
def save_processed_data(df):
    # Export cleaned water main break records
    # Export cleaned water main break records as timestamped JSONL
    output_dir = ROOT_DIR / "data" / "processed" / "hospitals"
    output_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    output_file = output_dir / f"hospitals_clean_{timestamp}.jsonl"

    df.to_json(
        output_file,
        orient="records",
        lines=True,
        date_format="iso",
        force_ascii=False
    )

    # Download file to local computer


    from google.colab import files as colab_files
    colab_files.download(str(output_file))

    print(f"Cleaned dataset saved: {output_file}")
    print(f"Exported records: {len(df)}")

In [67]:
save_processed_data(df_clean)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Cleaned dataset saved: /content/HYDROSIGHT/data/processed/hospitals/hospitals_clean_20261101T212247Z.jsonl
Exported records: 5


### Conclusion — Analyse exploratoire des hôpitaux

Le jeu de données comprend **5 établissements hospitaliers** situés à Calgary.

La validation géographique a confirmé que les établissements se trouvent à l'intérieur des limites municipales. Les cinq enregistrements présentent des coordonnées valides, sans doublons d'identifiants ni de noms.

Aucune exclusion ni transformation supplémentaire n'est nécessaire.

**Décision :** conserver les cinq établissements pour la priorisation géographique des zones à risque de rupture des conduites d'eau.

Ces données représentent une localisation des infrastructures hospitalières et ne seront pas utilisées comme preuve de leur présence historique pendant toute la période de modélisation.